# Chain of Responsibility Design Pattern 

explained using the classic Tech Support System example.

#### The Concept
The Chain of Responsibility passes a request along a chain of handlers. Upon receiving a request, each handler decides either to **process the request** or to **pass it to the next handler** in the chain.

**Analogy**: Calling Customer Service.
- **Level 1 Support (Bot)**: Can solve simple password resets. If not, passes to ->
- **Level 2 Support (Human)**: Can solve billing issues. If not, passes to ->
- **Level 3 Support (Manager)**: Can solve critical system failures.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we create an Abstract Handler that holds a reference (next_handler) to the next link in the chain. All concrete handlers inherit from this.

#### THE ABSTRACT HANDLER

In [1]:
from abc import ABC, abstractmethod
from typing import Optional

class SupportHandler(ABC):
    def __init__(self):
        self._next_handler: Optional[SupportHandler] = None

    def set_next(self, handler: 'SupportHandler') -> 'SupportHandler':
        self._next_handler = handler
        # Return the next handler to allow method chaining (handler.set_next().set_next())
        return handler

    @abstractmethod
    def handle_request(self, issue: str):
        # Base implementation: If I exist, pass to me. Otherwise, end of chain.
        if self._next_handler:
            return self._next_handler.handle_request(issue)
        return "End of Chain: No one could handle this request."

#### CONCRETE HANDLERS

In [2]:
class Level1Bot(SupportHandler):
    def handle_request(self, issue: str):
        if issue == "password_reset":
            return "Level 1 Bot: I reset your password. Have a nice day!"
        else:
            print("Level 1 Bot: I can't handle this. Passing to Human...")
            return super().handle_request(issue)

class Level2Human(SupportHandler):
    def handle_request(self, issue: str):
        if issue == "billing_error":
            return "Level 2 Human: I fixed the charge on your account."
        else:
            print("Level 2 Human: Too complex. Passing to Manager...")
            return super().handle_request(issue)

class Level3Manager(SupportHandler):
    def handle_request(self, issue: str):
        if issue == "system_crash":
            return "Level 3 Manager: Critical Team deployed. System rebooting."
        else:
            # If even the manager can't do it, we use the base class behavior
            print("Level 3 Manager: Even I don't know what to do.")
            return super().handle_request(issue)

#### CLIENT CODE

In [3]:
def main():
    # 1. Setup the chain
    bot = Level1Bot()
    human = Level2Human()
    manager = Level3Manager()

    # Link them: Bot -> Human -> Manager
    bot.set_next(human).set_next(manager)

    # 2. Test Cases
    print("--- Client: My password is forgotten ---")
    print(bot.handle_request("password_reset"))
    print("\n--- Client: I have a billing error ---")
    print(bot.handle_request("billing_error"))
    print("\n--- Client: The Server is on Fire ---")
    print(bot.handle_request("system_crash"))
    print("\n--- Client: I want a pizza ---")
    print(bot.handle_request("order_pizza"))

if __name__ == "__main__":
    main()

--- Client: My password is forgotten ---
Level 1 Bot: I reset your password. Have a nice day!

--- Client: I have a billing error ---
Level 1 Bot: I can't handle this. Passing to Human...
Level 2 Human: I fixed the charge on your account.

--- Client: The Server is on Fire ---
Level 1 Bot: I can't handle this. Passing to Human...
Level 2 Human: Too complex. Passing to Manager...
Level 3 Manager: Critical Team deployed. System rebooting.

--- Client: I want a pizza ---
Level 1 Bot: I can't handle this. Passing to Human...
Level 2 Human: Too complex. Passing to Manager...
Level 3 Manager: Even I don't know what to do.
End of Chain: No one could handle this request.


## The Pythonic Way

In Python, we can simplify this using **Coroutines (Generators) or Functions**. There is no need for a complex Class hierarchy if we just want a pipeline of logic. We can iterate through a list of functions until one returns a result.

Here is a version using **Functions + Iterables** (Middleware style).

#### THE HANDLERS (Simple Functions)

Each handler takes an 'issue' and returns a string (Success) or None (Pass)

In [4]:
from typing import Optional

def bot_handler(issue: str) -> Optional[str]:
    if issue == "password_reset":
        return "🤖 Bot: Reset link sent."
    print("🤖 Bot: Passing...")
    return None

def human_handler(issue: str) -> Optional[str]:
    if issue == "billing_error":
        return "👨‍💼 Human: Refund processed."
    print("👨‍💼 Human: Passing...")
    return None

def manager_handler(issue: str) -> Optional[str]:
    if issue == "system_crash":
        return "👩‍💻 Manager: Rebooting servers."
    print("👩‍💻 Manager: Don't know this issue.")
    return None

#### THE CHAIN EXECUTOR

In [5]:
from typing import List, Callable, Optional

class SupportChain:
    def __init__(self):
        # The chain is just a list of callables
        self.handlers: List[Callable[[str], Optional[str]]] = []

    def add_handler(self, handler_func):
        self.handlers.append(handler_func)

    def process(self, issue: str):
        # Iterate through functions. Stop at the first one that returns a value.
        for handle in self.handlers:
            result = handle(issue)
            if result:
                return result
        return "❌ Error: Request could not be handled."

#### CLIENT CODE

In [6]:
def main():
    # Setup
    chain = SupportChain()
    chain.add_handler(bot_handler)
    chain.add_handler(human_handler)
    chain.add_handler(manager_handler)

    # Test
    requests = ["password_reset", "billing_error", "system_crash", "order_pizza"]

    for req in requests:
        print(f"Requesting: {req}")
        result = chain.process(req)
        print(f"Result: {result}\n")

if __name__ == "__main__":
    main()

Requesting: password_reset
Result: 🤖 Bot: Reset link sent.

Requesting: billing_error
🤖 Bot: Passing...
Result: 👨‍💼 Human: Refund processed.

Requesting: system_crash
🤖 Bot: Passing...
👨‍💼 Human: Passing...
Result: 👩‍💻 Manager: Rebooting servers.

Requesting: order_pizza
🤖 Bot: Passing...
👨‍💼 Human: Passing...
👩‍💻 Manager: Don't know this issue.
Result: ❌ Error: Request could not be handled.



#### Key Differences

| Feature      | Classic OOP                                      | Pythonic                                                |
|--------------|--------------------------------------------------|----------------------------------------------------------|
| **Structure** | Rigid inheritance (`Handler` class).             | Simple list of functions.                               |
| **Linking**   | Linked-list style (`next_handler`).              | Iteration style (`for handler in list`).                 |
| **Flexibility** | Dynamic reordering is harder (must re-link).     | Dynamic reordering is easy (list pop/insert).            |
| **Use Case**  | Complex logic requiring internal state.          | Middleware, simple logic pipelines (like web frameworks). |


# Chain of Responsibility Pattern 

explained with a complex, real-world example: A Web Server Request Pipeline.

**The Scenario: Middleware Pipeline**

When a request hits a web server (like handling a payment), it must pass through several layers of security and validation before the actual business logic runs:
- **Authentication**: Is the user logged in?
- **Authorization (RBAC)**: Does the user have **"Admin"** rights?
- **Throttling**: Has the user made too many requests?
- **Validation**: Is the payment data valid?

If any layer fails, the chain stops immediately and returns an error.

## The Classic OOP Way (Java-Style)

In this approach, every handler is a Class. Each handler holds a reference (next_handler) to the next step in the chain.

#### ABSTRACT HANDLER

In [7]:
from abc import ABC, abstractmethod
from typing import Dict, Any, Optional

class Middleware(ABC):
    def __init__(self):
        self._next: Optional['Middleware'] = None

    def link_with(self, next_handler: 'Middleware') -> 'Middleware':
        """
        Builds the chain. Returns the next handler so we can chain calls:
        handler.link_with(b).link_with(c)
        """
        self._next = next_handler
        return next_handler

    @abstractmethod
    def handle(self, request: Dict[str, Any]) -> str:
        """
        Subclasses implement logic here.
        If success, they MUST call super().handle(request) to continue the chain.
        """
        if self._next:
            return self._next.handle(request)
        return "✅ Success: Request fully processed."

#### CONCRETE HANDLERS

In [8]:
class Authentication(Middleware):
    def handle(self, request: Dict[str, Any]) -> str:
        print("1. [Auth] Checking credentials...")
        if request.get("email") != "admin@company.com":
            return "❌ Error: 401 Unauthorized"
        return super().handle(request)

class Throttling(Middleware):
    def __init__(self, request_limit: int):
        super().__init__()
        self.limit = request_limit

    def handle(self, request: Dict[str, Any]) -> str:
        print("2. [Throttling] Checking request rate...")
        if request.get("req_count", 0) > self.limit:
            return "❌ Error: 429 Too Many Requests"
        return super().handle(request)

class DataValidation(Middleware):
    def handle(self, request: Dict[str, Any]) -> str:
        print("3. [Validation] Scanning payload...")
        if not request.get("payload"):
            return "❌ Error: 400 Bad Request (Empty Payload)"
        return super().handle(request)

#### CLIENT CODE

In [9]:
def main():
    # 1. Setup Chain
    auth = Authentication()
    throttle = Throttling(request_limit=5)
    validation = DataValidation()

    # Chain: Auth -> Throttle -> Validation
    # Note: We must start the execution from the HEAD (auth)
    auth.link_with(throttle).link_with(validation)

    # 2. Test Good Request
    print("--- Request 1 (Valid) ---")
    req_valid = {"email": "admin@company.com", "req_count": 2, "payload": "Buy Stuff"}
    print(auth.handle(req_valid))

    # 3. Test Bad Request (Fails at Step 1)
    print("\n--- Request 2 (Bad Auth) ---")
    req_bad_auth = {"email": "hacker@evil.com", "req_count": 1, "payload": "Steal"}
    print(auth.handle(req_bad_auth)) 
    # Notice: It never prints "[Throttling]..." because it stopped at Auth.

if __name__ == "__main__":
    main()

--- Request 1 (Valid) ---
1. [Auth] Checking credentials...
2. [Throttling] Checking request rate...
3. [Validation] Scanning payload...
✅ Success: Request fully processed.

--- Request 2 (Bad Auth) ---
1. [Auth] Checking credentials...
❌ Error: 401 Unauthorized


## The Pythonic Way (Functional / List Based)

In Python web frameworks (like Django, Flask, FastAPI), we rarely use linked-list classes. Instead, we use a List of Functions. This allows dynamic reordering and is much easier to read.

#### THE HANDLERS (Simple Functions)

In [10]:
from typing import Callable, List, Dict, Any, Optional

def check_auth(request: Dict[str, Any]) -> Optional[str]:
    print("--> [Middleware] Auth Check")
    if request.get("token") != "secure_token":
        return "401 Unauthorized"
    return None # Pass

def check_roles(request: Dict[str, Any]) -> Optional[str]:
    print("--> [Middleware] Role Check")
    if "admin" not in request.get("roles", []):
        return "403 Forbidden"
    return None # Pass

def check_sanity(request: Dict[str, Any]) -> Optional[str]:
    print("--> [Middleware] Sanity Check")
    if request.get("amount") < 0:
        return "400 Invalid Amount"
    return None # Pass

#### THE CHAIN ENGINE

In [11]:
class RequestPipeline:
    def __init__(self):
        # The chain is simply a list
        self.middlewares: List[Callable] = []

    def add(self, func: Callable):
        self.middlewares.append(func)

    def execute(self, request: Dict[str, Any]):
        # Iterate through the list.
        # If any function returns an error, STOP and return it.
        for step in self.middlewares:
            error = step(request)
            if error:
                return f"❌ Stopped at: {error}"
        
        # If loop finishes, business logic runs
        return self._business_logic(request)

    def _business_logic(self, request):
        return f"✅ Success: Transferred ${request['amount']}"

#### CLIENT CODE

In [12]:
def main():
    pipeline = RequestPipeline()
    
    # Dynamic Configuration
    pipeline.add(check_auth)
    pipeline.add(check_roles)
    # We can easily swap order or add new steps here
    pipeline.add(check_sanity) 

    # Test
    print("--- Test A: Hacker ---")
    req_hacker = {"token": "fake", "roles": [], "amount": 100}
    print(pipeline.execute(req_hacker))

    print("\n--- Test B: Valid Admin ---")
    req_admin = {"token": "secure_token", "roles": ["admin"], "amount": 500}
    print(pipeline.execute(req_admin))

if __name__ == "__main__":
    main()

--- Test A: Hacker ---
--> [Middleware] Auth Check
❌ Stopped at: 401 Unauthorized

--- Test B: Valid Admin ---
--> [Middleware] Auth Check
--> [Middleware] Role Check
--> [Middleware] Sanity Check
✅ Success: Transferred $500


#### Summary

- **Java Way**: Uses Inheritance (`extends Middleware`) and **Recursion**. Best if the handlers act like objects with internal state.
- **Python Way**: Uses Composition (List of Callables) and Iteration. Best for pipelines, data processing, and web middleware. It allows you to use simple functions instead of creating 10 different classes.